## ETL-проєкт

Задача від бізнесу:
У тебе є Excel-файл з таблицею продажів. Твоє завдання: автоматично зчитати дані, зробити певні розрахунки, очистити таблицю від зайвого, і завантажити її в базу даних SQLite.

Що треба зробити:
-	Зчитати Excel-файл sales.xlsx з read_excel() в ноутбук .ipynb
-	Додати колонку total_sum, де total_sum = quantity * price_per_unit * (1 - discount)
-	Видалити зайві колонки, залишити: order_id, product, quantity, total_sum, date
-	Записати результат у базу SQLite sales.db
-	Не обов’язково: Оформити весь код в одну функцію process_sales_to_sqlite() і створити запуск кожного дня о 9:00 з schedule.


In [1]:
import pandas as pd
import sqlite3
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Перевірка та завантаження файлу sales.xlsx з Google Drive
excel_file = '/content/drive/MyDrive/Data Analyst/ETL-проєкт/sales.xlsx'

# Перевіряємо наявність файлу
print(f"🔍 Шукаємо файл {excel_file}...")

if os.path.exists(excel_file):
    print(f"✅ Файл {excel_file} знайдено")

    # Завантажуємо файл
    df_sales = pd.read_excel(excel_file)
    initial_rows = len(df_sales)

    print(f"📖 Файл завантажено успішно")
    print(f"📊 Кількість рядків: {initial_rows}")
    print(f"📋 Кількість колонок: {len(df_sales.columns)}")
else:
    print(f"❌ Файл {excel_file} не знайдено!")

🔍 Шукаємо файл /content/drive/MyDrive/Data Analyst/ETL-проєкт/sales.xlsx...
✅ Файл /content/drive/MyDrive/Data Analyst/ETL-проєкт/sales.xlsx знайдено
📖 Файл завантажено успішно
📊 Кількість рядків: 30
📋 Кількість колонок: 6


In [4]:
# Перевірка необхідних колонок
required_columns = ['order_id', 'product', 'quantity', 'price_per_unit', 'discount', 'date']

print("🔍 Перевіряємо наявність необхідних колонок...")
print(f"Потрібні колонки: {required_columns}")

# Перевіряємо які колонки відсутні
missing_columns = [col for col in required_columns if col not in df_sales.columns]
available_columns = [col for col in required_columns if col in df_sales.columns]

if missing_columns:
    print(f"❌ Відсутні колонки: {missing_columns}")
    print(f"✅ Наявні колонки: {available_columns}")
    print("\n⚠️  Неможливо продовжити без всіх необхідних колонок!")

else:
    print(f"✅ Всі необхідні колонки присутні: {available_columns}")

    # Показуємо типи даних
    print(f"\n📊 Типи даних колонок:")
    for col in required_columns:
        print(f"   {col}: {df_sales[col].dtype}")

🔍 Перевіряємо наявність необхідних колонок...
Потрібні колонки: ['order_id', 'product', 'quantity', 'price_per_unit', 'discount', 'date']
✅ Всі необхідні колонки присутні: ['order_id', 'product', 'quantity', 'price_per_unit', 'discount', 'date']

📊 Типи даних колонок:
   order_id: object
   product: object
   quantity: int64
   price_per_unit: int64
   discount: float64
   date: datetime64[ns]


In [5]:
# Аналіз вихідних даних
if not missing_columns:
    print("📊 АНАЛІЗ ВИХІДНИХ ДАНИХ:")
    print("=" * 40)

    # Загальна інформація
    print(f"📋 Розмір даних: {df_sales.shape[0]} рядків × {df_sales.shape[1]} колонок")

    # Перевіряємо пропущені значення
    print(f"\n🔍 Пропущені значення:")
    null_counts = df_sales[required_columns].isnull().sum()

    if null_counts.sum() == 0:
        print("   ✅ Пропущених значень немає")
    else:
        for col, count in null_counts.items():
            if count > 0:
                print(f"   ⚠️  {col}: {count} пропущених значень")

    # Статистика по кількості та ціні
    print(f"\n📈 Статистика:")
    print(f"   Кількість (quantity): мін={df_sales['quantity'].min()}, макс={df_sales['quantity'].max()}, середнє={df_sales['quantity'].mean():.1f}")
    print(f"   Ціна (price_per_unit): мін=${df_sales['price_per_unit'].min():.2f}, макс=${df_sales['price_per_unit'].max():.2f}")
    print(f"   Знижка (discount): мін={df_sales['discount'].min():.1%}, макс={df_sales['discount'].max():.1%}")

    # Унікальні значення
    print(f"\n🏷️  Унікальні значення:")
    print(f"   Замовлення: {df_sales['order_id'].nunique()} унікальних")
    print(f"   Продукти: {df_sales['product'].nunique()} унікальних")
    print(f"   Продукти: {list(df_sales['product'].unique())}")

else:
    print("⏭️  Пропускаємо аналіз через відсутні колонки")

📊 АНАЛІЗ ВИХІДНИХ ДАНИХ:
📋 Розмір даних: 30 рядків × 6 колонок

🔍 Пропущені значення:
   ✅ Пропущених значень немає

📈 Статистика:
   Кількість (quantity): мін=1, макс=5, середнє=3.1
   Ціна (price_per_unit): мін=$400.00, макс=$1200.00
   Знижка (discount): мін=0.0%, макс=15.0%

🏷️  Унікальні значення:
   Замовлення: 30 унікальних
   Продукти: 5 унікальних
   Продукти: ['Кепка', 'Футболка', 'Кросівки', 'Рюкзак', 'Джинси']


In [6]:
# Обчислюємо total_sum
if not missing_columns:
    print("🧮 ОБЧИСЛЕННЯ TOTAL_SUM:")
    print("=" * 30)

    # Показуємо формулу
    print("📐 Формула: total_sum = quantity × price_per_unit × (1 - discount)")

    # Обчислюємо
    df_sales['total_sum'] = (df_sales['quantity'] *
                            df_sales['price_per_unit'] *
                            (1 - df_sales['discount']))

    print("✅ Колонка total_sum успішно додана")

    # Показуємо результат обчислень
    print(f"\n📊 Результати обчислень:")
    calculation_sample = df_sales[['order_id', 'product', 'quantity', 'price_per_unit', 'discount', 'total_sum']].head()
    display(calculation_sample)

    # Статистика по total_sum
    print(f"\n💰 Статистика total_sum:")
    print(f"   Загальна сума всіх продажів: ${df_sales['total_sum'].sum():.2f}")
    print(f"   Середня сума замовлення: ${df_sales['total_sum'].mean():.2f}")
    print(f"   Мінімальна сума: ${df_sales['total_sum'].min():.2f}")
    print(f"   Максимальна сума: ${df_sales['total_sum'].max():.2f}")

else:
    print("⏭️  Пропускаємо обчислення total_sum через відсутні колонки")

🧮 ОБЧИСЛЕННЯ TOTAL_SUM:
📐 Формула: total_sum = quantity × price_per_unit × (1 - discount)
✅ Колонка total_sum успішно додана

📊 Результати обчислень:


,order_id,product,quantity,price_per_unit,discount,total_sum
0,ORD1000,Кепка,1,600,0.1,540.0
1,ORD1001,Футболка,4,1000,0.1,3600.0
2,ORD1002,Кросівки,1,400,0.0,400.0
3,ORD1003,Кепка,2,400,0.1,720.0
4,ORD1004,Кепка,2,500,0.1,900.0



💰 Статистика total_sum:
   Загальна сума всіх продажів: $57225.00
   Середня сума замовлення: $1907.50
   Мінімальна сума: $380.00
   Максимальна сума: $4560.00


In [7]:
# Фільтрація колонок та очищення даних
if not missing_columns and 'total_sum' in df_sales.columns:
    print("🔍 ФІЛЬТРАЦІЯ ТА ОЧИЩЕННЯ ДАНИХ:")
    print("=" * 40)

    # Вибираємо потрібні колонки
    columns_to_keep = ['order_id', 'product', 'quantity', 'total_sum', 'date']
    print(f"📋 Залишаємо колонки: {columns_to_keep}")

    # Перевіряємо чи всі потрібні колонки є
    missing_filtered_cols = [col for col in columns_to_keep if col not in df_sales.columns]

    if missing_filtered_cols:
        print(f"❌ Відсутні колонки для фільтрації: {missing_filtered_cols}")
    else:
        # Фільтруємо колонки
        filtered_df_sales = df_sales[columns_to_keep].copy()

        print(f"✅ Колонки відфільтровано")
        print(f"📊 Розмір після фільтрації колонок: {filtered_df_sales.shape}")

        # Перевіряємо пропущені значення
        print(f"\n🧹 Очищення від пропущених значень:")
        before_cleaning = len(filtered_df_sales)
        null_counts_filtered = filtered_df_sales.isnull().sum()

        if null_counts_filtered.sum() == 0:
            print("   ✅ Пропущених значень немає - очищення не потрібне")
            after_cleaning = before_cleaning
        else:
            print("   ⚠️  Знайдено пропущені значення:")
            for col, count in null_counts_filtered.items():
                if count > 0:
                    print(f"      {col}: {count} пропущених")

            # Видаляємо рядки з пропущеними значеннями
            filtered_df_sales = filtered_df_sales.dropna()
            after_cleaning = len(filtered_df_sales)

            print(f"   🗑️  Видалено {before_cleaning - after_cleaning} рядків")

        print(f"\n📊 ПІДСУМОК ФІЛЬТРАЦІЇ:")
        print(f"   Початкових рядків: {initial_rows}")
        print(f"   Після очищення: {after_cleaning}")
        print(f"   Фінальних колонок: {len(filtered_df_sales.columns)}")

        # Показуємо фінальний результат
        print(f"\n📋 Перші 3 рядки відфільтрованих даних:")
        display(filtered_df_sales.head(3))

else:
    print("⏭️  Пропускаємо фільтрацію через попередні помилки")

🔍 ФІЛЬТРАЦІЯ ТА ОЧИЩЕННЯ ДАНИХ:
📋 Залишаємо колонки: ['order_id', 'product', 'quantity', 'total_sum', 'date']
✅ Колонки відфільтровано
📊 Розмір після фільтрації колонок: (30, 5)

🧹 Очищення від пропущених значень:
   ✅ Пропущених значень немає - очищення не потрібне

📊 ПІДСУМОК ФІЛЬТРАЦІЇ:
   Початкових рядків: 30
   Після очищення: 30
   Фінальних колонок: 5

📋 Перші 3 рядки відфільтрованих даних:


,order_id,product,quantity,total_sum,date
0,ORD1000,Кепка,1,540.0,2024-05-01
1,ORD1001,Футболка,4,3600.0,2024-05-02
2,ORD1002,Кросівки,1,400.0,2024-05-03


In [8]:
# Створення функції для типів даних SQLite
def map_dtype(dtype):

    if pd.api.types.is_integer_dtype(dtype):
        return "INTEGER"
    elif pd.api.types.is_float_dtype(dtype):
        return "REAL"
    elif pd.api.types.is_bool_dtype(dtype):
        return "INTEGER"
    else:
        return "TEXT"

# Тестуємо функцію
if 'filtered_df_sales' in locals():
    print("🔧 ФУНКЦІЯ ПЕРЕТВОРЕННЯ ТИПІВ ДАНИХ:")
    print("=" * 40)

    print("📊 Відповідність типів pandas → SQLite:")
    for col, dtype in filtered_df_sales.dtypes.items():
        sqlite_type = map_dtype(dtype)
        print(f"   {col}: {dtype} → {sqlite_type}")

    print("\n✅ Функція map_dtype готова до використання")
else:
    print("⚠️  filtered_df_sales не створено - функція готова, але не протестована")

🔧 ФУНКЦІЯ ПЕРЕТВОРЕННЯ ТИПІВ ДАНИХ:
📊 Відповідність типів pandas → SQLite:
   order_id: object → TEXT
   product: object → TEXT
   quantity: int64 → INTEGER
   total_sum: float64 → REAL
   date: datetime64[ns] → TEXT

✅ Функція map_dtype готова до використання


In [9]:
# Створюємо підключення
db_file = 'sales.db'
table_name = 'sales'

if 'filtered_df_sales' in locals() and len(filtered_df_sales) > 0:
    print("🗄️  ПІДКЛЮЧЕННЯ ДО БАЗИ ДАНИХ:")
    print("=" * 35)

    # Перевіряємо чи існує файл бази даних
    if os.path.exists(db_file):
        print(f"📁 База даних {db_file} вже існує")
        # Перевіряємо розмір файлу
        file_size = os.path.getsize(db_file)
        print(f"📊 Розмір файлу: {file_size} байт")
    else:
        print(f"🆕 Створюємо нову базу даних {db_file}")

    # Створюємо підключення
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    print(f"✅ Підключення до {db_file} успішне")

    # Перевіряємо існуючі таблиці
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    existing_tables = cursor.fetchall()

    if existing_tables:
        print(f"📋 Існуючі таблиці в базі:")
        for table in existing_tables:
            cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
            count = cursor.fetchone()[0]
            print(f"   📊 {table[0]}: {count} рядків")
    else:
        print("📋 База даних порожня - таблиць немає")

    print(f"\n🎯 Готуємося створити/оновити таблицю '{table_name}'")

else:
    print("⏭️  Пропускаємо створення підключення - дані не готові")

🗄️  ПІДКЛЮЧЕННЯ ДО БАЗИ ДАНИХ:
🆕 Створюємо нову базу даних sales.db
✅ Підключення до sales.db успішне
📋 База даних порожня - таблиць немає

🎯 Готуємося створити/оновити таблицю 'sales'


In [10]:
# Створюємо таблицю якщо підключення встановлено
if 'conn' in locals() and 'filtered_df_sales' in locals():
    print("🔨 СТВОРЕННЯ ТАБЛИЦІ:")
    print("=" * 25)

    # Генеруємо SQL типи для колонок
    print("🏗️  Визначаємо структуру таблиці")
    columns_with_types = []

    for col, dtype in filtered_df_sales.dtypes.items():
        sql_type = map_dtype(dtype)
        columns_with_types.append(f"{col} {sql_type}")

    # Створюємо SQL команду
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        {', '.join(columns_with_types)}
    )
    """

    # Виконуємо створення таблиці
    cursor.execute(create_table_sql)
    conn.commit()

    print(f"\n✅ Таблиця '{table_name}' успішно створена")

    # Перевіряємо структуру створеної таблиці
    cursor.execute(f"PRAGMA table_info({table_name})")
    table_info = cursor.fetchall()

    print(f"\n📋 Структура таблиці '{table_name}':")
    for info in table_info:
        col_id, col_name, col_type, not_null, default_val, primary_key = info
        print(f"   {col_name}: {col_type}")

else:
    print("⏭️  Пропускаємо створення таблиці - немає підключення або даних")

🔨 СТВОРЕННЯ ТАБЛИЦІ:
🏗️  Визначаємо структуру таблиці

✅ Таблиця 'sales' успішно створена

📋 Структура таблиці 'sales':
   order_id: TEXT
   product: TEXT
   quantity: INTEGER
   total_sum: REAL
   date: TEXT


In [11]:
# Завантаження даних в базу
print("📥 ЗАВАНТАЖЕННЯ ДАНИХ В БАЗУ:")
print("=" * 35)

# Перевіряємо поточний стан таблиці
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
current_count = cursor.fetchone()[0]
print(f"📊 Рядків в таблиці зараз: {current_count}")

if current_count > 0:
    print("⚠️  Таблиця не порожня!")
    print("🔄 Замінюємо існуючі дані новими ")
    if_exists_mode = 'replace'
else:
    print("✅ Таблиця порожня - додаємо нові дані")
    if_exists_mode = 'append'

 # Завантажуємо дані
rows_to_load = len(filtered_df_sales)
print(f"\n📤 Завантажуємо {rows_to_load} рядків...")

# Використовуємо to_sql для завантаження
filtered_df_sales.to_sql(table_name, conn, if_exists=if_exists_mode, index=False)

print(f"✅ Дані успішно завантажено в таблицю '{table_name}'")

# Перевіряємо результат
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
final_count = cursor.fetchone()[0]

print(f"\n📊 РЕЗУЛЬТАТ ЗАВАНТАЖЕННЯ:")
print(f"   Завантажено рядків: {rows_to_load}")
print(f"   Всього в таблиці: {final_count}")

# Показуємо приклад даних з бази
print(f"\n📋 Перші 3 рядки з таблиці '{table_name}':")
sample_query = f"SELECT * FROM {table_name} LIMIT 3"
sample_data = pd.read_sql_query(sample_query, conn)
display(sample_data)

📥 ЗАВАНТАЖЕННЯ ДАНИХ В БАЗУ:
📊 Рядків в таблиці зараз: 0
✅ Таблиця порожня - додаємо нові дані

📤 Завантажуємо 30 рядків...
✅ Дані успішно завантажено в таблицю 'sales'

📊 РЕЗУЛЬТАТ ЗАВАНТАЖЕННЯ:
   Завантажено рядків: 30
   Всього в таблиці: 30

📋 Перші 3 рядки з таблиці 'sales':


,order_id,product,quantity,total_sum,date
0,ORD1000,Кепка,1,540.0,2024-05-01 00:00:00
1,ORD1001,Футболка,4,3600.0,2024-05-02 00:00:00
2,ORD1002,Кросівки,1,400.0,2024-05-03 00:00:00


In [12]:
# Показуємо фінальну статистику
if 'conn' in locals():
    print("📈 ФІНАЛЬНА СТАТИСТИКА:")
    print("=" * 30)

    # Загальна інформація про таблицю
    info_query = f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT order_id) as unique_orders,
        COUNT(DISTINCT product) as unique_products,
        ROUND(SUM(total_sum), 2) as total_revenue,
        ROUND(AVG(total_sum), 2) as avg_order_value,
        ROUND(MIN(total_sum), 2) as min_order,
        ROUND(MAX(total_sum), 2) as max_order
    FROM {table_name}
    """

    stats_df = pd.read_sql_query(info_query, conn)

    print("📊 Загальна статистика:")
    print(f"   📋 Всього рядків: {stats_df['total_rows'].iloc[0]}")
    print(f"   🛒 Унікальних замовлень: {stats_df['unique_orders'].iloc[0]}")
    print(f"   📦 Унікальних продуктів: {stats_df['unique_products'].iloc[0]}")
    print(f"   💰 Загальна виручка: ${stats_df['total_revenue'].iloc[0]}")
    print(f"   📊 Середній чек: ${stats_df['avg_order_value'].iloc[0]}")
    print(f"   📉 Мінімальне замовлення: ${stats_df['min_order'].iloc[0]}")
    print(f"   📈 Максимальне замовлення: ${stats_df['max_order'].iloc[0]}")

    # Топ продуктів
    print(f"\n🏆 ТОП-3 ПРОДУКТИ ЗА СУМОЮ:")
    top_products_query = f"""
    SELECT
        product,
        COUNT(*) as orders_count,
        ROUND(SUM(total_sum), 2) as total_revenue
    FROM {table_name}
    GROUP BY product
    ORDER BY total_revenue DESC
    LIMIT 3
    """

    top_products = pd.read_sql_query(top_products_query, conn)
    display(top_products)

else:
    print("⚠️  Немає підключення до бази - статистика недоступна")

📈 ФІНАЛЬНА СТАТИСТИКА:
📊 Загальна статистика:
   📋 Всього рядків: 30
   🛒 Унікальних замовлень: 30
   📦 Унікальних продуктів: 5
   💰 Загальна виручка: $57225.0
   📊 Середній чек: $1907.5
   📉 Мінімальне замовлення: $380.0
   📈 Максимальне замовлення: $4560.0

🏆 ТОП-3 ПРОДУКТИ ЗА СУМОЮ:


,product,orders_count,total_revenue
0,Футболка,9,22150.0
1,Джинси,9,16335.0
2,Рюкзак,4,8175.0


In [13]:
# Закриваємо з'єднання та показуємо підсумок
if 'conn' in locals():
    # Закриваємо курсор та з'єднання
    if 'cursor' in locals():
        cursor.close()
        print("🔒 Курсор бази даних закрито")

    conn.close()
    print("🔒 З'єднання з базою даних закрито")

# Підсумок виконаної роботи
print(f"\n🎉 ЗАВДАННЯ ВИКОНАНО!")
print("=" * 25)

summary_points = [
    f"✅ Файл {'завантажено' if 'df_sales' in locals() else 'НЕ завантажено'}",
    f"✅ Колонки {'перевірено' if not locals().get('missing_columns', True) else 'НЕ всі знайдено'}",
    f"✅ total_sum {'обчислено' if 'total_sum' in locals().get('df_sales', pd.DataFrame()).columns else 'НЕ обчислено'}",
    f"✅ Дані {'відфільтровано' if 'filtered_df_sales' in locals() else 'НЕ відфільтровано'}",
    f"✅ Таблиця {'створена' if locals().get('create_table_sql') else 'НЕ створена'}",
    f"✅ Дані {'завантажено в БД' if locals().get('final_count', 0) > 0 else 'НЕ завантажено'}"
]

for point in summary_points:
    print(f"   {point}")

if 'filtered_df_sales' in locals():
    print(f"\n📊 Фінальний результат: {len(filtered_df_sales)} рядків збережено в {db_file}")
else:
    print(f"\n⚠️  Завдання виконано частково через помилки на попередніх етапах")

print(f"\n🎯 Файли створено:")
print(f"   📁 {db_file} - база даних SQLite")
if locals().get('final_count', 0) > 0:
    print(f"   📊 Таблиця '{table_name}' з даними всередині")

🔒 Курсор бази даних закрито
🔒 З'єднання з базою даних закрито

🎉 ЗАВДАННЯ ВИКОНАНО!
   ✅ Файл завантажено
   ✅ Колонки перевірено
   ✅ total_sum обчислено
   ✅ Дані відфільтровано
   ✅ Таблиця створена
   ✅ Дані завантажено в БД

📊 Фінальний результат: 30 рядків збережено в sales.db

🎯 Файли створено:
   📁 sales.db - база даних SQLite
   📊 Таблиця 'sales' з даними всередині


## Створення функції process_sales_to_sqlite()

In [14]:
!pip install -q schedule

In [15]:
import pandas as pd
import sqlite3
import os
import schedule
import time
from datetime import datetime

In [16]:
def map_dtype(dtype):
    """Функція для перетворення типів pandas в типи SQLite"""
    if pd.api.types.is_integer_dtype(dtype):
        return "INTEGER"
    elif pd.api.types.is_float_dtype(dtype):
        return "REAL"
    elif pd.api.types.is_bool_dtype(dtype):
        return "INTEGER"
    else:
        return "TEXT"

def process_sales_to_sqlite(excel_file='/content/drive/MyDrive/Data Analyst/ETL-проєкт/sales.xlsx', db_file='sales.db', table_name='sales'):
    """
    ETL функція для обробки файлу продажів та завантаження в SQLite

    Args:
        excel_file (str): Шлях до Excel файлу з даними продажів
        db_file (str): Шлях до SQLite бази даних
        table_name (str): Назва таблиці в базі даних

    Returns:
        bool: True якщо процес успішний, False якщо є помилки
    """

    try:
        print(f"\n🚀 ПОЧАТОК ETL ПРОЦЕСУ - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 60)

        # КРОК 1: Завантаження Excel файлу
        print(f"📁 КРОК 1: Завантаження файлу {excel_file}")

        if not os.path.exists(excel_file):
            print(f"❌ ПОМИЛКА: Файл {excel_file} не знайдено!")
            return False

        df_sales = pd.read_excel(excel_file)
        initial_rows = len(df_sales)
        print(f"✅ Файл завантажено: {initial_rows} рядків, {len(df_sales.columns)} колонок")

        # КРОК 2: Перевірка необхідних колонок
        print(f"\n🔍 КРОК 2: Перевірка колонок")
        required_columns = ['order_id', 'product', 'quantity', 'price_per_unit', 'discount', 'date']
        missing_columns = [col for col in required_columns if col not in df_sales.columns]

        if missing_columns:
            print(f"❌ ПОМИЛКА: Відсутні колонки: {missing_columns}")
            return False

        print(f"✅ Всі необхідні колонки присутні")

        # КРОК 3: Обчислення total_sum
        print(f"\n🧮 КРОК 3: Обчислення total_sum")
        df_sales['total_sum'] = (df_sales['quantity'] *
                                df_sales['price_per_unit'] *
                                (1 - df_sales['discount']))

        total_revenue = df_sales['total_sum'].sum()
        print(f"✅ total_sum обчислено. Загальна виручка: ${total_revenue:.2f}")

        # КРОК 4: Фільтрація колонок та очищення
        print(f"\n🔍 КРОК 4: Фільтрація та очищення даних")
        columns_to_keep = ['order_id', 'product', 'quantity', 'total_sum', 'date']

        # Фільтруємо колонки
        filtered_df_sales = df_sales[columns_to_keep].copy()

        # Видаляємо пропущені значення
        before_cleaning = len(filtered_df_sales)
        filtered_df_sales = filtered_df_sales.dropna()
        after_cleaning = len(filtered_df_sales)

        if before_cleaning != after_cleaning:
            print(f"🧹 Видалено {before_cleaning - after_cleaning} рядків з пропущеними значеннями")

        print(f"✅ Дані очищено: {after_cleaning} рядків готово до завантаження")

        # КРОК 5: Підключення до бази даних
        print(f"\n🗄️  КРОК 5: Підключення до бази даних")

        # Створюємо підключення
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()

        # Генеруємо SQL типи для колонок
        columns_with_types = []
        for col, dtype in filtered_df_sales.dtypes.items():
            sql_type = map_dtype(dtype)
            columns_with_types.append(f"{col} {sql_type}")

        # Створюємо таблицю
        create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {', '.join(columns_with_types)}
        )
        """

        cursor.execute(create_table_sql)
        conn.commit()
        print(f"✅ Таблиця '{table_name}' готова")

        # КРОК 6: Завантаження даних
        print(f"\n📥 КРОК 6: Завантаження даних в базу")

        # Перевіряємо поточний стан таблиці
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        current_count = cursor.fetchone()[0]

        if current_count > 0:
            print(f"🔄 Замінюємо {current_count} існуючих рядків")
            if_exists_mode = 'replace'
        else:
            print(f"📝 Додаємо дані в порожню таблицю")
            if_exists_mode = 'append'

        # Завантажуємо дані
        filtered_df_sales.to_sql(table_name, conn, if_exists=if_exists_mode, index=False)

        # Перевіряємо результат
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        final_count = cursor.fetchone()[0]

        print(f"✅ Завантажено {final_count} рядків в таблицю '{table_name}'")

        # КРОК 7: Фінальна статистика
        print(f"\n📊 КРОК 7: Фінальна статистика")

        stats_query = f"""
        SELECT
            COUNT(*) as total_rows,
            COUNT(DISTINCT order_id) as unique_orders,
            COUNT(DISTINCT product) as unique_products,
            ROUND(SUM(total_sum), 2) as total_revenue
        FROM {table_name}
        """

        stats_df = pd.read_sql_query(stats_query, conn)
        stats = stats_df.iloc[0]

        print(f"   📋 Всього рядків: {stats['total_rows']}")
        print(f"   🛒 Унікальних замовлень: {stats['unique_orders']}")
        print(f"   📦 Унікальних продуктів: {stats['unique_products']}")
        print(f"   💰 Загальна виручка: ${stats['total_revenue']}")

        # Закриваємо з'єднання
        cursor.close()
        conn.close()

        print(f"\n🎉 ETL ПРОЦЕС ЗАВЕРШЕНО УСПІШНО!")
        print(f"🕐 Час завершення: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 60)

        return True

    except Exception as e:
        print(f"\n❌ КРИТИЧНА ПОМИЛКА: {str(e)}")

        # Закриваємо з'єднання у випадку помилки
        if 'conn' in locals():
            try:
                if 'cursor' in locals():
                    cursor.close()
                conn.close()
                print("🔒 З'єднання з базою закрито після помилки")
            except:
                pass

        return False

def run_daily_etl():
    """Wrapper функція для щоденного запуску ETL"""
    print(f"\n⏰ ЩОДЕННИЙ ЗАПУСК ETL - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    success = process_sales_to_sqlite()

    if success:
        print("✅ Щоденний ETL процес завершено успішно")
    else:
        print("❌ Щоденний ETL процес завершено з помилками")

    return success

def setup_daily_schedule():
    """Налаштування щоденного розкладу"""

    print("⏰ НАЛАШТУВАННЯ АВТОМАТИЧНОГО ЗАПУСКУ")
    print("=" * 40)

    # Налаштовуємо щоденний запуск о 9:00
    schedule.every().day.at("09:00").do(run_daily_etl)

    print("✅ Розклад налаштовано:")
    print("   🕘 Щодня о 09:00 - process_sales_to_sqlite()")
    print("   📅 Початок роботи: завтра о 09:00")

    # Показуємо наступний запуск
    next_run = schedule.next_run()
    print(f"   ⏭️  Наступний запуск: {next_run.strftime('%Y-%m-%d %H:%M:%S')}")

    return True



In [17]:
# Тестуємо функцію
print("🧪 ТЕСТОВИЙ ЗАПУСК ETL ФУНКЦІЇ:")
print("=" * 35)

# Запускаємо ETL процес
success = process_sales_to_sqlite()

if success:
    print("\n🎯 НАЛАШТУВАННЯ АВТОМАТИЧНОГО ЗАПУСКУ:")
    print("=" * 45)

    # Налаштовуємо розклад
    setup_daily_schedule()

    print(f"\n💡 ДЛЯ ПОСТІЙНОЇ РОБОТИ ПЛАНУВАЛЬНИКА:")
    print("   Запустіть окремий Python скрипт з кодом:")
    print("   ")
    print("   while True:")
    print("       schedule.run_pending()")
    print("       time.sleep(60)")

else:
    print("\n❌ Функція не працює - перевірте дані та спробуйте ще раз")

🧪 ТЕСТОВИЙ ЗАПУСК ETL ФУНКЦІЇ:

🚀 ПОЧАТОК ETL ПРОЦЕСУ - 2026-09-04 19:05:23
📁 КРОК 1: Завантаження файлу /content/drive/MyDrive/Data Analyst/ETL-проєкт/sales.xlsx
✅ Файл завантажено: 30 рядків, 6 колонок

🔍 КРОК 2: Перевірка колонок
✅ Всі необхідні колонки присутні

🧮 КРОК 3: Обчислення total_sum
✅ total_sum обчислено. Загальна виручка: $57225.00

🔍 КРОК 4: Фільтрація та очищення даних
✅ Дані очищено: 30 рядків готово до завантаження

🗄️  КРОК 5: Підключення до бази даних
✅ Таблиця 'sales' готова

📥 КРОК 6: Завантаження даних в базу
🔄 Замінюємо 30 існуючих рядків
✅ Завантажено 30 рядків в таблицю 'sales'

📊 КРОК 7: Фінальна статистика
   📋 Всього рядків: 30.0
   🛒 Унікальних замовлень: 30.0
   📦 Унікальних продуктів: 5.0
   💰 Загальна виручка: $57225.0

🎉 ETL ПРОЦЕС ЗАВЕРШЕНО УСПІШНО!
🕐 Час завершення: 2026-09-04 19:05:23

🎯 НАЛАШТУВАННЯ АВТОМАТИЧНОГО ЗАПУСКУ:
⏰ НАЛАШТУВАННЯ АВТОМАТИЧНОГО ЗАПУСКУ
✅ Розклад налаштовано:
   🕘 Щодня о 09:00 - process_sales_to_sqlite()
   📅 Початок роботи: